In [ ]:
# ==========================================
# TESIS ZMVM: PM, clima y salud
# Autor: Arely Leal
# Descripción: procesa los resultados de NowCast de PM₁₀ y PM₂.₅ y clasifica la calidad del aire calculando el porcentaje mensual de días por categoría con base en los criterios estipulados en la NOM-172-SEMARNAT-2019. 
# Periodo: 2000-2019
# ==========================================

In [ ]:
#Importar librerías
import pandas as pd
import numpy as np
import os

#Configuración de archivos
PM10_CSV = "AGREGAR RUTA DEL ARCHIVO"
PM25_CSV = "AGREGAR RUTA DEL ARCHIVO"

OUT_DIR_PM10 = os.path.dirname(PM10_CSV)
OUT_DIR_PM25 = os.path.dirname(PM25_CSV)

#Umbrales NowCast

THRESHOLDS = {
    "PM10": [
        ("Buena", 0, 50, True, True),
        ("Aceptable", 50, 75, False, True),
        ("Mala", 75, 155, False, True),
        ("Muy Mala", 155, 235, False, True),
        ("Extremadamente Mala", 235, np.inf, False, True),
    ],
    "PM25": [
        ("Buena", 0, 25, True, True),
        ("Aceptable", 25, 45, False, True),
        ("Mala", 45, 79, False, True),
        ("Muy Mala", 79, 147, False, True),
        ("Extremadamente Mala", 147, np.inf, False, True),
    ],
}

CATEGORIES = ["Buena", "Aceptable", "Mala", "Muy Mala", "Extremadamente Mala"]
SIN_DATOS = "Sin datos"

#Formato Datetime
DATETIME_FORMAT = "%d/%m/%y %H:%M"

#Sumar 100%
PCT_DECIMALS = 2

#Funciones auxiliares
def clean_numeric(series: pd.Series) -> pd.Series:
    series = series.replace(["NaN", "nan", ""], np.nan)
    return pd.to_numeric(series, errors="coerce")

def classify_value(x: float, pollutant_key: str):
    if pd.isna(x):
        return np.nan
    for name, lo, hi, lo_incl, hi_incl in THRESHOLDS[pollutant_key]:
        lo_ok = (x >= lo) if lo_incl else (x > lo)
        hi_ok = (x <= hi) if hi_incl else (x < hi)
        if lo_ok and hi_ok:
            return name
    return np.nan

def force_sum_100(df: pd.DataFrame, pct_cols: list, residual_col: str, decimals: int) -> pd.DataFrame:
    # Redondear todos
    for c in pct_cols:
        df[c] = df[c].round(decimals)

    others = [c for c in pct_cols if c != residual_col]
    df[residual_col] = (100.0 - df[others].sum(axis=1)).round(decimals)

    # Evitar -0.00 / negativos por casos raros
    df[residual_col] = df[residual_col].replace(-0.0, 0.0)
    df.loc[df[residual_col] < 0, residual_col] = 0.0

    df["pct_suma"] = df[pct_cols].sum(axis=1).round(decimals)
    return df

def month_base_from_daily(daily: pd.DataFrame) -> pd.DataFrame:
    """
    Base mensual por municipio, con días calendario del mes.
    Se usa para asegurar que exista una fila por (municipio, mes),
    incluso si no hubo ningún día con dato válido en ese mes.
    """
    base = (
        daily.groupby(["municipio", "month", "month_period", "days_in_month"], as_index=False)
             .size()
             .drop(columns=["size"])
    )
    return base

def process_one(pollutant_key: str, csv_path: str, value_col: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    df["datetime"] = pd.to_datetime(df["datetime"], format=DATETIME_FORMAT, errors="coerce")
    df = df.dropna(subset=["datetime"]).copy()

    # Limpieza numérica del NowCast (convierte "NaN" texto a NaN real)
    df[value_col] = clean_numeric(df[value_col])

    # Auxiliares
    df["hour"] = df["datetime"].dt.hour
    df["date"] = df["datetime"].dt.date
    df["month_period"] = df["datetime"].dt.to_period("M")
    df["month"] = df["month_period"].astype(str)
    df["days_in_month"] = df["month_period"].dt.days_in_month

    outputs = []

    for target_hour in [8, 18]:
        dfi = df[df["hour"] == target_hour].copy()

# Un valor por día para cada hora objetivo (08:00 y 18:00),
        daily = (
            dfi.groupby(["municipio", "month_period", "month", "days_in_month", "date"], as_index=False)[value_col]
               .max()
               .rename(columns={value_col: "nowcast"})
        )

        daily["categoria"] = daily["nowcast"].apply(lambda x: classify_value(x, pollutant_key))

        # Conteos por categoría (solo días con categoría válida)
        counts = (
            daily.dropna(subset=["categoria"])
                 .groupby(["municipio", "month", "month_period", "days_in_month", "categoria"], as_index=False)
                 .agg(dias=("date", "nunique"))
        )

        # Base mensual (asegura fila por municipio-mes)
        base = month_base_from_daily(daily)

        # Pivot de conteos
        wide = counts.pivot_table(
            index=["municipio", "month", "month_period", "days_in_month"],
            columns="categoria",
            values="dias",
            fill_value=0
        ).reset_index()

        # Merge con base para incluir meses sin datos válidos
        outm = base.merge(wide, on=["municipio", "month", "month_period", "days_in_month"], how="left")

        # Rellenar NaN en categorías con 0
        for c in CATEGORIES:
            if c not in outm.columns:
                outm[c] = 0
            outm[c] = outm[c].fillna(0).astype(int)

        # Días calendario
        outm["dias_mes"] = outm["days_in_month"].astype(int)

        # Días con dato válido (clasificados)
        outm["dias_con_dato"] = outm[CATEGORIES].sum(axis=1)

        # Días sin datos (para esa hora)
        outm[SIN_DATOS] = (outm["dias_mes"] - outm["dias_con_dato"]).clip(lower=0).astype(int)

        # Porcentajes base
        for c in CATEGORIES:
            outm[f"pct_{c}"] = (outm[c] / outm["dias_mes"]) * 100.0
        outm["pct_Sin_datos"] = (outm[SIN_DATOS] / outm["dias_mes"]) * 100.0

        # Suma 100% exacta
        pct_cols = [f"pct_{c}" for c in CATEGORIES] + ["pct_Sin_datos"]
        outm = force_sum_100(outm, pct_cols=pct_cols, residual_col="pct_Sin_datos", decimals=PCT_DECIMALS)

        # Salida final
        final = outm[
            ["municipio", "month", "dias_mes"]
            + CATEGORIES + [SIN_DATOS]
            + [f"pct_{c}" for c in CATEGORIES] + ["pct_Sin_datos", "pct_suma"]
        ].copy()

        final.insert(0, "contaminante", pollutant_key)
        final.insert(1, "hora_objetivo", target_hour)

        outputs.append(final)

    return pd.concat(outputs, ignore_index=True)

pm10_res = process_one("PM10", PM10_CSV, "nowcast_PM10_12h")
pm25_res = process_one("PM25", PM25_CSV, "nowcast_PM25_12h")


#Guardar resultados
out_pm10 = os.path.join(OUT_DIR_PM10, "porcentaje_dias_mes_calidad_PM10_por_hora_SUM100.csv")
out_pm25 = os.path.join(OUT_DIR_PM25, "porcentaje_dias_mes_calidad_PM25_por_hora_SUM100.csv")

pm10_res.to_csv(out_pm10, index=False)
pm25_res.to_csv(out_pm25, index=False)

print("Archivos generados")
print(" - PM10 :", out_pm10)
print(" - PM2.5:", out_pm25)

print("Filas PM10:", pm10_res.shape[0], "| Filas PM2.5:", pm25_res.shape[0])
